In [7]:
import pandas as pd
from pathlib import Path
import os

from src.models.demonstracoes_contabeis import DemonstracoesContabeis

In [8]:
data = DemonstracoesContabeis()
data.get_anos_disponiveis()
data.get_demonstracoes_contabeis()
data.save_demonstracoes_contabeis()

## Lendo csv

In [9]:
file = "1T2025.csv"
df_1T = pd.read_csv(f"{Path(os.getcwd()).resolve().parent}/data/raw/demonstracoes_contabeis/{file}",
      sep=";",           
      encoding="latin-1",
      decimal=",")
df_1T.head()

,DATA,REG_ANS,CD_CONTA_CONTABIL,DESCRICAO,VL_SALDO_INICIAL,VL_SALDO_FINAL
0,2025-01-01,316458,46411,PUBLICIDADE E PROPAGANDA,0.0,1070.00
1,2025-01-01,316458,464119,Publicidade e Propaganda,0.0,1070.00
2,2025-01-01,316458,46411901,Publicidade e Propaganda,0.0,1070.00
3,2025-01-01,316458,464119011,Publicidade e Propaganda,0.0,1070.00
4,2025-01-01,316458,465,DESPESAS COM TRIBUTOS,0.0,99024.72


In [10]:
file_2 = "2T2025.csv"
df_2T = pd.read_csv(f"{Path(os.getcwd()).resolve().parent}/data/raw/demonstracoes_contabeis/{file_2}",
      sep=";",           
      encoding="latin-1",
      decimal=",")
df_2T.head()

,DATA,REG_ANS,CD_CONTA_CONTABIL,DESCRICAO,VL_SALDO_INICIAL,VL_SALDO_FINAL
0,2025-04-01,421723,452119014,Juros - PESL,0.0,0.0
1,2025-04-01,421723,452119015,VariaÃ§Ã£o cambial - PESL,0.0,0.0
2,2025-04-01,421723,452119019,Outras,0.0,0.0
3,2025-04-01,421723,45212,DESPESAS FINANCEIRAS COM OPERAÃÃES DE ASSIST...,0.0,0.0
4,2025-04-01,421723,452129,Despesas Financeiras com OperaÃ§Ãµes de Assist...,0.0,0.0


In [11]:
file_3 = "3T2025.csv"
df_3T = pd.read_csv(f"{Path(os.getcwd()).resolve().parent}/data/raw/demonstracoes_contabeis/{file_2}",
      sep=";",           
      encoding="latin-1",
      decimal=",")
df_3T.head()

,DATA,REG_ANS,CD_CONTA_CONTABIL,DESCRICAO,VL_SALDO_INICIAL,VL_SALDO_FINAL
0,2025-04-01,421723,452119014,Juros - PESL,0.0,0.0
1,2025-04-01,421723,452119015,VariaÃ§Ã£o cambial - PESL,0.0,0.0
2,2025-04-01,421723,452119019,Outras,0.0,0.0
3,2025-04-01,421723,45212,DESPESAS FINANCEIRAS COM OPERAÃÃES DE ASSIST...,0.0,0.0
4,2025-04-01,421723,452129,Despesas Financeiras com OperaÃ§Ãµes de Assist...,0.0,0.0


In [12]:
df_reg_ans = data.get_relatorios_cadop()
df_reg_ans.head()

,REG_ANS,CNPJ,Razao_Social
0,410942,3607971000192,24 HORAS SISTEMA DE SAUDE S/C LTDA
1,422908,41788751000100,3S ADMINISTRADORA DE BENEFICIOS LTDA
2,419575,18108766000150,A LA SANTE ADMINISTRADORA DE BENEFICIO LTDA.
3,320439,50928563000112,A M P ASSISTENCIA MEDICA PAULISTA S/C LTDA
4,406821,3246912000136,A ORAL OESTE'S ASSISTENCIA EM ODONTOLOGIA S/C ...


In [13]:
cnpjs_duplicados = df_reg_ans.groupby('CNPJ')['Razao_Social'].nunique()
cnpjs_com_razoes_distintas = cnpjs_duplicados[cnpjs_duplicados > 1]


cnpjs_problema = cnpjs_com_razoes_distintas.index.tolist()
df_problema = df_reg_ans.loc[df_reg_ans['CNPJ'].isin(cnpjs_problema)].sort_values('CNPJ')
df_problema.head()

,REG_ANS,CNPJ,Razao_Social
1639,413402,126507000196,MASSA FALIDA DE UNILIFE SAÃDE LTDA.
1868,40349,126507000196,ODONTO SAÃDE LTDA
1153,304620,175304000190,GRALHA AZUL SERVIÃOS DE SAUDE S/C LTDA.
1152,299,175304000190,GRALHA AZUL SAÃDE S.A.
724,37377,655209000193,CONSORCIO DE ALUMINIO DO MARANHAO


Nesse caso nos deparamos com a empresas que de forma geral mudaram de razão social, é possível inferir baseado no fato de haverem similaridades nas razões sociais, portanto manteremos ambos os casos, que correspondem à mesma empresa, também podendo ter valores distintos de REG_ANS devido ao fato de poderem ter encerrados seus convenios e reaberto posteriormente

## Processando os dados

In [14]:
df_concat = pd.concat([df_1T, df_2T, df_3T], ignore_index=True)
df_concat.head()

,DATA,REG_ANS,CD_CONTA_CONTABIL,DESCRICAO,VL_SALDO_INICIAL,VL_SALDO_FINAL
0,2025-01-01,316458,46411,PUBLICIDADE E PROPAGANDA,0.0,1070.00
1,2025-01-01,316458,464119,Publicidade e Propaganda,0.0,1070.00
2,2025-01-01,316458,46411901,Publicidade e Propaganda,0.0,1070.00
3,2025-01-01,316458,464119011,Publicidade e Propaganda,0.0,1070.00
4,2025-01-01,316458,465,DESPESAS COM TRIBUTOS,0.0,99024.72


Aqui todos em memória mesmo, mas poderia filtrar em lotes, no caso como foi para a análise eu preferi fazer isso para ser mais prático, no programa eu farei dessa forma também, para melhor agilidade do código, e por saber que os arquivos ainda com o overhead do pandas não superariam 2gb, cosia que não causaria um problema de memória

In [15]:
df_filtrado = df_concat[df_concat.DESCRICAO.str.contains(r'(?=.*eventos)(?=.*sinistros)', 
    case=False, 
    na=False, 
    regex=True)]
df_filtrado.head()

,DATA,REG_ANS,CD_CONTA_CONTABIL,DESCRICAO,VL_SALDO_INICIAL,VL_SALDO_FINAL
97,2025-01-01,316849,131719011,DepÃ³sitos Judiciais - Eventos / Sinistros,289349.17,292907.23
105,2025-01-01,316849,21111203,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,0.00,0.00
132,2025-01-01,316849,23111202,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,203169.04,199868.05
133,2025-01-01,316849,231112022,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,203169.04,199868.05
152,2025-01-01,316903,231111021,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,25535.00,25535.00


In [16]:
df_filrado_com_cnpj = df_filtrado.merge(df_reg_ans, how="inner", on="REG_ANS")
df_filrado_com_cnpj.head()

,DATA,REG_ANS,CD_CONTA_CONTABIL,DESCRICAO,VL_SALDO_INICIAL,VL_SALDO_FINAL,CNPJ,Razao_Social
0,2025-01-01,316849,131719011,DepÃ³sitos Judiciais - Eventos / Sinistros,289349.17,292907.23,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL
1,2025-01-01,316849,21111203,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,0.00,0.00,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL
2,2025-01-01,316849,23111202,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,203169.04,199868.05,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL
3,2025-01-01,316849,231112022,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,203169.04,199868.05,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL
4,2025-01-01,316903,231111021,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,25535.00,25535.00,93507895000136,POLIMÃDICA SAÃDE SOCIEDADE SIMPLES LTDA


In [17]:
df_filrado_com_cnpj["ValorDespesas"] = df_filrado_com_cnpj.VL_SALDO_FINAL - df_filrado_com_cnpj.VL_SALDO_INICIAL
df_filrado_com_cnpj.head()

,DATA,REG_ANS,CD_CONTA_CONTABIL,DESCRICAO,VL_SALDO_INICIAL,VL_SALDO_FINAL,CNPJ,Razao_Social,ValorDespesas
0,2025-01-01,316849,131719011,DepÃ³sitos Judiciais - Eventos / Sinistros,289349.17,292907.23,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL,3558.06
1,2025-01-01,316849,21111203,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,0.00,0.00,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL,0.00
2,2025-01-01,316849,23111202,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,203169.04,199868.05,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL,-3300.99
3,2025-01-01,316849,231112022,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,203169.04,199868.05,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL,-3300.99
4,2025-01-01,316903,231111021,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,25535.00,25535.00,93507895000136,POLIMÃDICA SAÃDE SOCIEDADE SIMPLES LTDA,0.00


In [18]:
cols = ['CNPJ', 'Razao_Social', 'ValorDespesas']
df_fim = df_filrado_com_cnpj[cols]
df_fim = df_fim.rename(columns={
    'Razao_Social' : 'RazaoSocial'
})
df_fim.head()

,CNPJ,RazaoSocial,ValorDespesas
0,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL,3558.06
1,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL,0.00
2,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL,-3300.99
3,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL,-3300.99
4,93507895000136,POLIMÃDICA SAÃDE SOCIEDADE SIMPLES LTDA,0.00


Em contas de eventos/sinistros, valores negativos podem ocorrer por alguns motivos:

Glosas e estornos: Quando há devolução de valores pagos anteriormente (glosas de operadoras, correções de lançamentos)
Provisões revertidas: Se havia uma provisão no saldo inicial que foi revertida no período
Transferências entre contas: Reclassificações contábeis entre diferentes contas de sinistros
Créditos recebidos: Ressarcimentos, recuperações de sinistros, ou acordos com prestadores

ou seja, esses valores podem de fato ocorrer, sem serem problemas associados ao nosso banco de dados!